# Session 02: Python Language Essentials

**Notebook 1 of 3: Concepts** · used during the live session and for revision afterwards (about 50 minutes on your own)

You already know variables, conditions, loops and collections. This notebook skips what they *are* and focuses on how Python *behaves*: the places where your Java, C# or TypeScript instincts give the right answer, and the places where they quietly give the wrong one.

By the end of this notebook you will be able to:

1. **Predict** what happens when two names refer to the same object, and **avoid** aliasing bugs with the right kind of copy.
2. **Explain** the difference between mutable and immutable objects, and why only some objects can be dict keys.
3. **Apply** truthiness, `None` checks and `is` versus `==` correctly, including the cases where truthiness is the wrong tool.
4. **Write** loops with `enumerate` and `zip` instead of index counters, and **replace** filter-and-transform loops with comprehensions.
5. **Use** slicing to take, skip and reverse parts of sequences.

**Then continue with:** `03-collections-deep-dive.ipynb` (lists, tuples, dicts and sets in depth, and a full data transformation), and finally the self-check quiz in `02-practice-quiz.ipynb`.

> **Kernel:** run this notebook on the course environment's kernel, the `.venv` in the course repository, exactly as in Session 01. Everything here uses only the standard library.

## The engineering problem

Back at **Orbital Logistics** (fictional), the platform team's on-call dashboard is built from a nightly export of incident tickets. Two bugs have been reported this week:

1. After the "mark as reviewed" step ran, the **raw** export also showed every ticket as reviewed. Nobody wrote to the raw data, yet it changed.
2. The "unprioritised tickets" count includes tickets whose priority is `0`, the highest level in the legacy system. They *have* a priority.

Neither bug would compile away in Java or C#. Both come from Python rules that look familiar but differ in the details:

- Assignment in Python **never copies**. It attaches another name to the same object.
- Python decides "true or false" for *any* value, not just booleans, and `0` counts as false.

This notebook makes those rules explicit, so you can predict this behaviour instead of discovering it in production.

## Part 1: Names and objects

Here is the mental model that explains most of this session: **a Python variable is a name attached to an object, not a box that holds a value.**

If you come from Java or C#, this is already how reference types (objects, arrays, `List<T>`) work. The difference is that in Python *everything* is an object, including integers, strings and `None`. There are no primitives stored "inside" the variable.

![Names refer to objects](../../../assets/diagrams/python-object-references.svg)

**How this works**

- `ticket = {...}` creates a dict object and attaches the name `ticket` to it. The object lives on its own; the name is just a label in the current namespace.
- `alias = ticket` copies nothing. It attaches a second label to the **same** object, so a change made through `alias` is visible through `ticket`. There is only one dict.
- `ticket.copy()` builds a **new** outer dict, but fills it with the same references the old one held. The `tags` list is not copied, so both dicts point at one shared list.
- That's why a "copy" can still leak changes: modifying `snapshot["tags"]` changes what `ticket["tags"]` shows. Replacing a key (`snapshot["status"] = "closed"`) does *not*, because that rebinds a slot in the new dict only.
- When nested data must be fully independent, use `copy.deepcopy()`, which copies every level. It's slower and rarely needed, so reach for it deliberately.

### Dynamic typing, but strong typing

A name has no type; the **object** does. Rebinding a name to an object of a different type is legal:

In [ ]:
priority = "P1"
print(type(priority).__name__, priority)

priority = 1
print(type(priority).__name__, priority)

That's **dynamic** typing. It does not mean Python converts types for you, which is where it differs from JavaScript. Python is **strongly** typed: mixing incompatible types raises an error instead of guessing.

**Predict:** JavaScript gives `"3" + 3 === "33"`. What does Python do?

In [ ]:
try:
    print("3" + 3)
except TypeError as error:
    print("TypeError:", error)

**Explain:** Python refuses to guess. You convert explicitly: `int("3") + 3` or `"3" + str(3)`.

| | Java / C# | TypeScript (compiled to JS) | Python |
|---|---|---|---|
| Who has the type? | The variable (declared) and the object | The variable (at compile time); values at run time | Only the object |
| Rebind to another type? | No (compile error) | Compile error, but JS allows it | Yes |
| `"3" + 3` | `"33"` (string concatenation) | `"33"` | `TypeError` |
| Primitives stored in the variable? | Yes (`int`, `double`, `bool`) | n/a | No: everything is an object |

Type hints such as `priority: int = 1`, which you met in Session 01, document intent for your editor and tools. They do not change any of this behaviour at run time.

### Aliasing in action

`id()` returns an object's identity, a number that is unique among objects that exist at the same time. Two names with the same `id` refer to the same object. The `is` operator compares identities directly.

**Predict:** after this cell runs, what is `ticket["status"]`?

In [ ]:
ticket = {"id": "INC-1042", "status": "open", "tags": ["network"]}
alias = ticket

alias["status"] = "reviewed"

print("ticket:", ticket)
print("Same object?", alias is ticket, "| ids:", id(alias) == id(ticket))

**Explain:** there was only ever one dict. This is bug 1 from the engineering problem: the "review" step received the raw export, not a copy of it.

Now compare a shallow copy with a deep copy, the two situations in the diagram:

In [ ]:
import copy

ticket = {"id": "INC-1042", "status": "open", "tags": ["network"]}

snapshot = ticket.copy()        # shallow: new outer dict, same inner list
independent = copy.deepcopy(ticket)  # deep: new dict AND new list

snapshot["status"] = "closed"        # rebinds a key in the copy only
snapshot["tags"].append("db")        # mutates the list both dicts share

print("ticket:     ", ticket)
print("snapshot:   ", snapshot)
print("independent:", independent)
print("Shared tags list?", snapshot["tags"] is ticket["tags"])

**Explain:** `ticket["status"]` is still `"open"`, because assigning to a key of `snapshot` changed only the new dict. But `ticket["tags"]` now contains `"db"`, because the list was shared. `independent` is untouched.

The same three options exist for lists: `alias = items`, `items.copy()` (or `list(items)`, or `items[:]`), and `copy.deepcopy(items)`.

**Modify:** change `snapshot["tags"].append("db")` to `snapshot["tags"] = ["db"]` and re-run. **Explain** why `ticket` is no longer affected.

### Passing objects to functions

Python passes arguments the same way it assigns names: the parameter becomes **another name for the caller's object**. This is often called "pass by object reference" or "pass by assignment". If you know Java, it behaves exactly like passing an object reference: the function can mutate the object, but rebinding the parameter has no effect outside.

**Predict:** which of these two functions changes the caller's list?

In [ ]:
def add_escalation_tag(tags):
    tags.append("escalated")   # mutates the object the caller passed in


def replace_tags(tags):
    tags = ["escalated"]       # rebinds the local name only


incident_tags = ["network"]
add_escalation_tag(incident_tags)
print("after add_escalation_tag:", incident_tags)

incident_tags = ["network"]
replace_tags(incident_tags)
print("after replace_tags:      ", incident_tags)

Ruff (Session 04) flags the line in `replace_tags` with `F841 Local variable 'tags' is assigned to but never used`. That warning is a useful hint here: an assignment that nothing reads afterwards can't have any effect.

**Engineering Insight:** a function that mutates its arguments surprises its callers. The convention in Python code is: if a function changes its input, it returns `None` (like `list.sort()`). If it returns a new value, it leaves the input alone (like `sorted()`). Follow the same convention in your own functions and readers can tell which kind they're calling from the return type alone.

## Part 2: Mutable and immutable objects

Some objects can change after they're created (**mutable**), and some can't (**immutable**). In Java terms, every Python `str`, `int` and `tuple` behaves like `java.lang.String`: operations that look like changes actually produce new objects.

![Mutable versus immutable objects](../../../assets/diagrams/mutable-vs-immutable.svg)

**How this works**

- On the left, `open_count += 1` can't change the integer `5`, because integers are immutable. Python computes a new object, `6`, and rebinds `open_count` to it. `previous` still refers to the original `5`.
- On the right, `tags.append("db")` changes the one list object in place. Every name attached to that list (`tags` and `same_tags`) sees the change, because there is nothing else to see.
- Aliasing is therefore only *dangerous* with mutable objects. Sharing an immutable object between ten names is perfectly safe, since nobody can change it.
- The footer is worth memorising: `int`, `float`, `bool`, `str`, `None`, `tuple`, `frozenset` and `bytes` are immutable; `list`, `dict`, `set` and most class instances are mutable.
- Mutability also decides what can be a dict key or set member. Python looks keys up by their **hash**, and a hash must never change, so mutable objects are rejected.

### Strings are immutable

**Predict:** what does this print?

In [ ]:
team = "  Platform "
team.strip().lower()
print(repr(team))

clean_team = team.strip().lower()
print(repr(clean_team))

**Explain:** string methods never change the string; they **return** a new one. The first call's result was thrown away. This is a frequent mistake for anyone used to in-place APIs, and nothing warns you about it.

(`repr()` shows a value the way you'd type it in code, quotes and spaces included, which makes whitespace visible. It's the best way to print strings while debugging.)

### The `+=` surprise

For immutable types, `x += y` always builds a new object. For a **list**, `+=` is defined to extend the list *in place*. So the same-looking line behaves differently depending on the type.

**Predict:** what is `on_call_backup` after each block?

In [ ]:
on_call = ["amira"]
on_call_backup = on_call
on_call += ["jonas"]         # list +=  -> extends in place
print("after +=:     ", on_call_backup)

on_call = ["amira"]
on_call_backup = on_call
on_call = on_call + ["jonas"]  # +  -> builds a new list, then rebinds
print("after = ... +:", on_call_backup)

**Explain:** `+=` mutated the shared list, so the backup changed. `on_call + [...]` created a new list and rebound only `on_call`, so the backup kept the old one. You rarely need to think about this, until two names share a list.

### Hashable: what can be a key

Tuples are immutable, so they work as dict keys; that's very handy for composite keys such as `(team, priority)`. Lists don't.

In [ ]:
breach_counts = {("platform", "P1"): 2, ("database", "P2"): 1}
print(breach_counts[("platform", "P1")])

try:
    bad_key = {["platform", "P1"]: 2}
except TypeError as error:
    print("TypeError:", error)

**Common Mistake:** a tuple is only hashable if everything inside it is. `("platform", ["network"])` is a tuple, but it contains a list, so `hash()` on it raises the same `unhashable type: 'list'` error. The tuple can't change which objects it holds, but the list inside it can still change.

## Part 3: Numbers, strings, booleans and `None`

These are the types you'll use constantly. You know the concepts already; here are the differences that matter.

### Numbers

| | Java / C# | JavaScript | Python |
|---|---|---|---|
| Integer size | fixed (32/64-bit, overflows) | all numbers are 64-bit floats | `int` is **arbitrary precision**: never overflows |
| `7 / 2` | `3` (integer division) | `3.5` | `3.5` (always a float) |
| Integer division | `7 / 2` | `Math.floor(7 / 2)` | `7 // 2` → `3` |
| Exact decimals | `BigDecimal` / `decimal` | libraries | `decimal.Decimal` |

**Predict** each line before you run the cell:

In [ ]:
print(7 / 2, 7 // 2, 7 % 2)
print(-7 // 2)            # floor division rounds DOWN, towards negative infinity
print(2 ** 100)           # no overflow
print(0.1 + 0.2 == 0.3)   # binary floating point, same as every other language
print(round(2.5), round(3.5))

**Explain:**

- `-7 // 2` is `-4`, not `-3`: Python's `//` floors, while Java and C# truncate towards zero. This matters when you bucket negative offsets or time differences.
- `0.1 + 0.2 == 0.3` is `False` everywhere, not just in Python. Compare floats with `math.isclose()`, and use `decimal.Decimal` for money.
- `round()` uses **banker's rounding** (round half to even), so `round(2.5)` is `2`. That's fine for statistics, and a surprise in invoices.

### Strings

Strings are immutable sequences of Unicode characters. Single and double quotes are equivalent. For building strings, use **f-strings**; they're the modern, readable choice, and they support format specifications after a colon:

In [ ]:
ticket_id = "INC-1042"
age_hours = 26.456
team = "platform"

print(f"{ticket_id} has been open for {age_hours:.1f} hours (team: {team!r})")
print(f"|{team:<10}|{team:>10}|{team:^10}|")   # left, right, centre in 10 columns
print(f"{1234567:,}")                          # thousands separator
print(f"{age_hours=}")                         # 'self-documenting' form, handy for debugging

The handful of string methods you'll use in almost every data task:

In [ ]:
raw = "  INC-1042 | Platform | P1  "

parts = [part.strip() for part in raw.split("|")]
print(parts)
print(" / ".join(parts))
print(parts[0].startswith("INC-"), "form" in parts[1], parts[1].lower())
print(raw.replace("|", ","))

**Engineering Insight:** `"sep".join(items)` looks backwards to most newcomers, because the separator is the object. It's also the efficient way to build a big string. Repeated `text += piece` in a loop creates a new string each time; collecting pieces in a list and joining once avoids that.

### Booleans and `None`

- `True` and `False` are capitalised. The operators are words: `and`, `or`, `not`.
- `None` is Python's single "no value" object, like `null`. There is no separate `undefined`.
- `bool` is a subclass of `int`, so `True + True == 2`. That's occasionally handy (`sum(ticket["breached"] for ticket in tickets)` counts the `True`s) and occasionally surprising.

## Part 4: Truthiness, `None`, and `is` versus `==`

### Truthiness

Every Python object can be used as a condition. These are **falsy**; almost everything else is truthy:

| Falsy values | |
|---|---|
| `None`, `False` | |
| Zero of any numeric type | `0`, `0.0`, `Decimal("0")` |
| Empty collections and strings | `""`, `[]`, `()`, `{}`, `set()`, `range(0)` |

That makes "is there anything?" checks concise:

In [ ]:
open_tickets = []

if not open_tickets:
    print("Nothing open. Quiet night.")

assignee = ""
print("assignee:", assignee or "unassigned")

`a or b` returns `a` if it's truthy, otherwise `b`. It returns one of the *operands*, not `True`/`False`. That makes `value or default` a common idiom.

**Comparison with JavaScript:** the falsy lists are similar, except for one trap. In JavaScript, an empty array `[]` and empty object `{}` are **truthy**. In Python, empty collections are **falsy**. If you write TypeScript daily, `if (items)` and `if items` do not mean the same thing.

### When truthiness is the wrong tool

Truthiness merges "missing" with "present but zero/empty". That's bug 2 from the engineering problem.

**Predict:** which tickets does this code report as unprioritised?

In [ ]:
tickets = [
    {"id": "INC-1", "priority": 0},     # 0 = highest priority in the legacy system
    {"id": "INC-2", "priority": None},  # genuinely missing
    {"id": "INC-3", "priority": 2},
]

print("Buggy:  ", [t["id"] for t in tickets if not t["priority"]])
print("Correct:", [t["id"] for t in tickets if t["priority"] is None])

**Rule of thumb:** use truthiness when "empty" and "missing" really mean the same thing (an empty list of tickets, a blank name). When `0`, `""` or `False` are valid values, test for `None` explicitly with `is None`.

### `is` versus `==`

- `==` asks "do these have **equal values**?" It is what you want almost every time, and it's like `.equals()` in Java or `Equals()` in C#.
- `is` asks "are these the **same object**?" Use it for `None` (`x is None`, `x is not None`), and for sentinels you create yourself. Not for numbers or strings.

In [ ]:
first_export = ["INC-1", "INC-2"]
second_export = ["INC-1", "INC-2"]

print("equal values: ", first_export == second_export)
print("same object:  ", first_export is second_export)

threshold = 1000
parsed = int("1000")   # e.g. read from a config file
print("1000 == 1000: ", threshold == parsed)
print("1000 is 1000: ", threshold is parsed)

**Explain:** the two lists have equal contents but are two separate objects. The integer case is worse: `threshold is parsed` happens to be `False` here, but CPython caches small integers (-5 to 256), so the same comparison with `5` would print `True`. Code that uses `is` on numbers or strings gives results that depend on interpreter details. Ruff flags it, and so does Python itself if you write `is` next to a literal (`SyntaxWarning: "is" with 'int' literal`).

**Comparison with Java:** this is the reverse of the Java string trap. In Java, `==` compares references and you must remember `.equals()`. In Python, `==` already compares values; `is` is the rarely needed identity check.

### Chained comparisons

Python lets you chain comparisons the way you'd write them in maths:

In [ ]:
age_hours = 30
print(24 < age_hours <= 72)          # same as: 24 < age_hours and age_hours <= 72
print(1 <= len("P1") < 3)

## Part 5: Conditions and loops

### Conditions

`if` / `elif` / `else` work as you'd expect. Blocks are defined by **indentation** (4 spaces by convention), not braces. The one-line conditional expression reads in a different order from `?:`:

In [ ]:
age_hours = 30
sla_hours = 24

# value_if_true if condition else value_if_false
status = "BREACHED" if age_hours > sla_hours else "within SLA"
print(status)

Python also has a `match` statement (structural pattern matching, since 3.10). It's more powerful than a `switch`, and you'll see it in some codebases, but plain `if`/`elif` or a dict lookup covers most needs. We won't use `match` in this course.

### Loops iterate over items, not indexes

The biggest style shift for C-family developers: a `for` loop in Python is always a "for each". You loop over the **items**. If you catch yourself writing `range(len(...))`, there's almost always a clearer form.

In [ ]:
team_names = ["platform", "database", "network"]

# The habit to unlearn:
for i in range(len(team_names)):
    print(i, team_names[i])

Need the position too? Use `enumerate()`, which hands you `(index, item)` pairs. Need two lists in step? Use `zip()`:

In [ ]:
for position, team in enumerate(team_names, start=1):
    print(f"{position}. {team}")

on_call_engineers = ["amira", "jonas", "li"]
for team, engineer in zip(team_names, on_call_engineers, strict=True):
    print(f"{team:<10} -> {engineer}")

`zip()` normally stops silently at the end of the **shorter** input. `strict=True` raises a `ValueError` instead when the lengths differ, which is what you want when a mismatch means bad data. Use it unless you actually want the truncation.

Dicts iterate over their **keys**; use `.items()` to get keys and values together, and unpack them straight into two names:

In [ ]:
sla_hours = {"P1": 4, "P2": 24, "P3": 72}

for priority, hours in sla_hours.items():
    print(f"{priority}: respond within {hours} h")

`while` loops, `break` and `continue` work as in other languages. `range(start, stop, step)` produces numbers lazily, and like slicing (Part 7), `stop` is excluded.

### Debugging: changing a list while looping over it

**Predict:** the goal is to drop every closed ticket. What's left?

In [ ]:
statuses = ["closed", "closed", "open", "closed", "open"]

for status in statuses:
    if status == "closed":
        statuses.remove(status)

print(statuses)

**Observe:** a `"closed"` survived. There was no error, just a wrong answer.

**Hypothesize and inspect:** the loop walks the list by position. Removing item 0 shifts everything left, so the item that moved into position 0 is never visited. Add `print(status, statuses)` as the first line in the loop to watch it happen.

**Fix:** never add to or remove from a list while iterating over it. Build a new list instead, which is exactly what a comprehension does (Part 6):

In [ ]:
statuses = ["closed", "closed", "open", "closed", "open"]
still_open = [status for status in statuses if status != "closed"]
print(still_open)

**Verify:** no `"closed"` remains, and the original list is unchanged if anything else still needs it. (Python doesn't raise an error for this mistake with lists. It does for dicts: changing a dict's size during iteration raises `RuntimeError: dictionary changed size during iteration`.)

## Part 6: Comprehensions

A huge share of everyday data code is "take a collection, keep some items, transform them". Python's syntax for that is the **comprehension**. You've seen it already; here it is properly.

```text
[ expression   for item in iterable   if condition ]
  └ transform    └ loop                  └ filter (optional)
```

Read it as: *"build a list of `expression`, for each `item` in `iterable`, if `condition`."*

In [ ]:
tickets = [
    {"id": "INC-1001", "team": "platform", "priority": "P1", "age_hours": 6},
    {"id": "INC-1002", "team": "database", "priority": "P3", "age_hours": 80},
    {"id": "INC-1003", "team": "platform", "priority": "P2", "age_hours": 30},
    {"id": "INC-1004", "team": "network", "priority": "P1", "age_hours": 2},
]

# Loop version
p1_ids = []
for ticket in tickets:
    if ticket["priority"] == "P1":
        p1_ids.append(ticket["id"])
print(p1_ids)

# Comprehension version: same result
p1_ids = [ticket["id"] for ticket in tickets if ticket["priority"] == "P1"]
print(p1_ids)

The same syntax with braces builds a **set** or a **dict**, and with parentheses inside a function call it becomes a **generator expression**, which computes items one at a time without building a list:

In [ ]:
teams = {ticket["team"] for ticket in tickets}                     # set: unique teams
age_by_id = {ticket["id"]: ticket["age_hours"] for ticket in tickets}  # dict: id -> age
total_age = sum(ticket["age_hours"] for ticket in tickets)          # generator: no list built

print(sorted(teams))
print(age_by_id)
print(total_age)
print(any(ticket["age_hours"] > 72 for ticket in tickets))          # stops at the first True

| Task | Java streams | C# LINQ | JavaScript | Python |
|---|---|---|---|---|
| Filter and map | `.filter(...).map(...).toList()` | `.Where(...).Select(...).ToList()` | `.filter(...).map(...)` | `[f(x) for x in xs if p(x)]` |
| To a set | `.collect(toSet())` | `.ToHashSet()` | `new Set(...)` | `{f(x) for x in xs}` |
| To a map | `.collect(toMap(...))` | `.ToDictionary(...)` | `Object.fromEntries(...)` | `{k(x): v(x) for x in xs}` |
| Sum / any | `.mapToInt(...).sum()` / `.anyMatch(...)` | `.Sum(...)` / `.Any(...)` | `.reduce(...)` / `.some(...)` | `sum(...)` / `any(...)` |

**When *not* to use a comprehension:**

- When the loop exists for its **side effects** (printing, writing files, calling APIs). `[print(x) for x in xs]` builds a useless list of `None`s; write a `for` loop.
- When it no longer fits comfortably on one or two lines, or nests more than two `for` clauses. A plain loop with good names is easier to debug, and you can put a breakpoint inside it.

**Modify:** write a comprehension for the IDs of tickets older than 24 hours, then a dict comprehension mapping each ticket ID to `"BREACHED"` or `"ok"` using a 24-hour limit. (Hint: the conditional expression from Part 5 works inside a comprehension's expression.)

In [ ]:
# Your turn: replace the two `None`s.
older_than_a_day = None
sla_flags = None

print(older_than_a_day)
print(sla_flags)

## Part 7: Slicing

Lists, tuples and strings support **slicing**: `sequence[start:stop:step]`. `start` is included, `stop` is excluded, and any part can be left out. Negative numbers count from the end.

```text
 index:     0       1       2       3       4       5
         ["INC-1","INC-2","INC-3","INC-4","INC-5","INC-6"]
 from end: -6      -5      -4      -3      -2      -1
```

In [ ]:
recent = ["INC-1", "INC-2", "INC-3", "INC-4", "INC-5", "INC-6"]

print(recent[0], recent[-1])   # first, last
print(recent[:3])              # first three
print(recent[-2:])             # last two
print(recent[2:4])             # positions 2 and 3
print(recent[::2])             # every second item
print(recent[::-1])            # reversed copy
print("INC-1042"[4:])          # strings slice too

Three properties that make slicing safe and useful:

- **A slice always returns a new object**, so `recent[:]` is a shallow copy of a list (Part 1).
- **Out-of-range slices don't raise**: `recent[:100]` simply returns everything, while `recent[100]` raises `IndexError`. That's convenient for "top N" when there may be fewer than N.
- `len(sequence[a:b]) == b - a` when both are in range, the same half-open convention as `range()`.

**Predict, then run:** what do `recent[-3:-1]`, `recent[4:2]` and `recent[1:-1]` return?

In [ ]:
print(recent[-3:-1])
print(recent[4:2])
print(recent[1:-1])

**Explain:** `[-3:-1]` is positions 3 and 4. `[4:2]` is empty, because with the default step of 1 you can't go backwards. `[1:-1]` drops the first and last items, a handy "trim the ends" idiom (for example to skip a header and footer row).

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| `b = a` to "copy" a list or dict | Both names share one object; changes leak | `a.copy()`, or `copy.deepcopy(a)` for nested data |
| `.copy()` on nested data | Inner lists and dicts are still shared | `copy.deepcopy()`, or build new inner objects |
| Calling a string method and ignoring the result | Nothing changes; strings are immutable | `name = name.strip()` |
| `if not value:` when `0` or `""` is valid | Valid values treated as missing | `if value is None:` |
| `x is 1000`, `name is "platform"` | Result depends on interpreter caching | `==` for values; `is` only for `None` and sentinels |
| `for i in range(len(items))` | Noisy, and index bugs creep in | `for item in items`, `enumerate()`, `zip()` |
| Removing items from a list while looping over it | Items skipped silently | Build a new list with a comprehension |
| `zip()` over lists of different lengths | Extra items dropped silently | `zip(..., strict=True)` |
| Using a list as a dict key | `TypeError: unhashable type: 'list'` | Use a tuple |
| A comprehension for side effects | Builds a throwaway list of `None` | A plain `for` loop |

## Practical engineering application

Every later session moves data like this: JSON from files (Session 03), HTTP responses (Session 05), validated models (Session 06), LLM outputs (Session 09) and document chunks (Session 11). The rules from this notebook decide whether that code is correct:

- **Don't mutate data you were given.** Return new objects, so the caller's data is safe and your functions are easy to test.
- **Be explicit about missing values.** `is None` checks prevent a whole category of "valid zero treated as missing" bugs.
- **Prefer comprehensions for transformations, loops for actions.**

## Key takeaways

- A variable is a **name attached to an object**. Assignment and argument passing never copy.
- Mutability decides whether sharing is dangerous: mutate a shared list and every name sees it; immutable objects can't change at all.
- Only hashable (in practice, immutable) objects can be dict keys or set members. Tuples are the usual composite key.
- Python is dynamically **and strongly** typed: no implicit `"3" + 3`.
- Empty collections, `0`, `""` and `None` are falsy. Use `is None` when zero or empty is a legitimate value.
- `==` compares values; `is` compares identity and is for `None` and sentinels.
- Loop over items; use `enumerate()` and `zip(strict=True)` for positions and parallel sequences; use comprehensions for filter-and-transform.
- Slices are half-open, never raise for out-of-range bounds, and return new objects.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** continue with **`03-collections-deep-dive.ipynb`**, where you'll choose between list, tuple, dict and set, and put all of this together in a real data transformation. Then test yourself with **`02-practice-quiz.ipynb`**.